In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import os, json, csv, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

LOGS_DIR = "../logs"   # relative to notebooks/
PIVOT_CSV = os.path.join(LOGS_DIR, "ablation_pivot.csv")
print("Logs dir:", os.path.abspath(LOGS_DIR))


In [ ]:
# ── Load aggregated results ────────────────────────────────────────────────
# Run `python scripts/aggregate_results.py` first to generate ablation_pivot.csv
if not os.path.exists(PIVOT_CSV):
    print(f"[!] {PIVOT_CSV} not found. Running aggregation now …")
    os.system(f"python ../scripts/aggregate_results.py --logs_dir {LOGS_DIR}")

df = pd.read_csv(PIVOT_CSV)
print(f"Loaded {len(df)} rows, columns: {list(df.columns)}")
df.head(10)


In [ ]:
# ── Performance Summary Table ──────────────────────────────────────────────
METRIC_DISPLAY = {
    "mAP_coco":   "mAP@[.5:.95]",
    "mAP_50":     "mAP@0.5",
    "mAP_75":     "mAP@0.75",
    "mAP_small":  "mAP_small",
    "mAP_medium": "mAP_medium",
    "mAP_large":  "mAP_large",
    "AR_1":       "AR@1",
    "AR_10":      "AR@10",
}
pivot = df.pivot_table(
    index=["run", "stage"],
    values=list(METRIC_DISPLAY.keys()),
    aggfunc="first",
).round(2)
pivot.columns = [METRIC_DISPLAY[c] for c in pivot.columns]
pivot.sort_index(inplace=True)
print("Full performance table (values in %):")
pivot


In [ ]:
# ── Ablation A: mAP vs. Number of Refinement Stages (T=1,2,3) ─────────────
# Uses the 'default' run which was evaluated at stages 1, 2, and 3.
abl_a = df[df["run"] == "default"].sort_values("stage")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# mAP@0.5 and mAP@[.5:.95]
for ax, metric, label, color in [
    (axes[0], "mAP_50",   "mAP@0.5",       "#2196F3"),
    (axes[1], "mAP_coco", "mAP@[0.5:0.95]","#FF5722"),
]:
    if metric in abl_a.columns:
        ax.bar(abl_a["stage"].astype(str), abl_a[metric], color=color, alpha=0.85, width=0.5)
        for i, (_, row) in enumerate(abl_a.iterrows()):
            ax.text(i, row[metric] + 0.3, f"{row[metric]:.2f}%", ha="center", fontsize=10)
    ax.set_xlabel("Cascade Stage (T)")
    ax.set_ylabel(f"{label} (%)")
    ax.set_title(f"Ablation A — {label} vs. T")
    ax.set_ylim(0, max(abl_a[metric].max() * 1.15 if metric in abl_a.columns else 60, 10))

plt.suptitle("Ablation A: Accuracy vs. Number of Refinement Stages", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, "ablation_A_stages.png"), bbox_inches="tight")
plt.show()
print("Saved ablation_A_stages.png")


In [ ]:
# ── Ablation B: With vs. Without Centerness at Refinement Stages ──────────
# Requires 'default' and 'abl_B_no_cent' runs in the pivot CSV.
runs_needed = ["default", "abl_B_no_cent"]
abl_b = df[df["run"].isin(runs_needed)].copy()

if len(abl_b["run"].unique()) < 2:
    print("[!] Run 'abl_B_no_cent' not yet available. "
          "Run evaluate.py with --no_centerness_stages 2 3 --run_name abl_B_no_cent first.")
else:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    stage_labels = sorted(abl_b["stage"].unique())
    x      = np.arange(len(stage_labels))
    width  = 0.35
    colors = {"default": "#4CAF50", "abl_B_no_cent": "#F44336"}
    labels = {"default": "With centerness", "abl_B_no_cent": "No centerness (stages 2,3)"}

    for i, (run, color) in enumerate(colors.items()):
        sub = abl_b[abl_b["run"] == run].sort_values("stage")
        if sub.empty:
            continue
        bars = ax.bar(x + i * width, sub["mAP_50"], width, label=labels[run],
                      color=color, alpha=0.85)
        for bar, val in zip(bars, sub["mAP_50"]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                    f"{val:.2f}%", ha="center", fontsize=9)

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([f"Stage {s}" for s in stage_labels])
    ax.set_ylabel("mAP@0.5 (%)")
    ax.set_title("Ablation B — Effect of Centerness on mAP@0.5", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(LOGS_DIR, "ablation_B_centerness.png"), bbox_inches="tight")
    plt.show()
    print("Saved ablation_B_centerness.png")


In [ ]:
# ── Ablation C: Center-Sampling Radius Schedule ────────────────────────────
# Compares: default (progressive 1.5→1.0→0.75),
#           abl_C_uniform (1.5→1.5→1.5),
#           abl_C_aggressive (1.5→0.75→0.5)
runs_c  = ["default", "abl_C_uniform", "abl_C_aggressive"]
labels_c = {
    "default":          "Progressive (1.5→1.0→0.75)",
    "abl_C_uniform":    "Uniform (1.5→1.5→1.5)",
    "abl_C_aggressive": "Aggressive (1.5→0.75→0.5)",
}
colors_c = {"default": "#2196F3", "abl_C_uniform": "#FF9800", "abl_C_aggressive": "#9C27B0"}
abl_c = df[df["run"].isin(runs_c)].copy()

available = abl_c["run"].unique()
if len(available) < 2:
    print(f"[!] Need at least 2 runs from {runs_c}. "
          "Train and evaluate abl_C variants first (see configs/ablations/README.md).")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for ax, metric, ylabel in [
        (axes[0], "mAP_50",   "mAP@0.5 (%)"),
        (axes[1], "mAP_coco", "mAP@[.5:.95] (%)"),
    ]:
        for run in runs_c:
            sub = abl_c[abl_c["run"] == run].sort_values("stage")
            if sub.empty:
                continue
            ax.plot(sub["stage"], sub[metric], marker="o", linewidth=2,
                    label=labels_c[run], color=colors_c[run])
        ax.set_xlabel("Cascade Stage")
        ax.set_ylabel(ylabel)
        ax.set_xticks([1, 2, 3])
        ax.set_title(f"Ablation C — {ylabel}")
        ax.legend(fontsize=9)

    plt.suptitle("Ablation C: Effect of Center-Sampling Radius Schedule",
                 fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(LOGS_DIR, "ablation_C_radius.png"), bbox_inches="tight")
    plt.show()
    print("Saved ablation_C_radius.png")


In [ ]:
# ── Complexity: Latency vs. mAP Scatter ────────────────────────────────────
# Shows accuracy/speed trade-off for each experiment variant.
# Only rows with measured latency (full_cascade_ms) are shown.
cplx = df.dropna(subset=["ms_per_img"]).copy()

if cplx.empty:
    print("[!] No complexity data found. Run evaluate.py without --skip_complexity.")
else:
    fig, ax = plt.subplots(figsize=(8, 5))
    for _, row in cplx.iterrows():
        ax.scatter(row["ms_per_img"], row["mAP_50"], s=120, zorder=3)
        ax.annotate(
            f"{row['run']} (S{int(row['stage'])})",
            (row["ms_per_img"], row["mAP_50"]),
            textcoords="offset points", xytext=(6, 4), fontsize=8
        )
    ax.set_xlabel("Inference Latency (ms / image)")
    ax.set_ylabel("mAP@0.5 (%)")
    ax.set_title("Accuracy vs. Speed Trade-Off", fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(LOGS_DIR, "complexity_scatter.png"), bbox_inches="tight")
    plt.show()
    print("Saved complexity_scatter.png")


In [ ]:
# ── Inference Overhead Bonus Check ─────────────────────────────────────────
# Checks whether the full cascade adds <15% latency vs. stage-1-only.
default_row = df[(df["run"] == "default") & (df["stage"] == 3)]

if "overhead_pct" not in default_row.columns or default_row["overhead_pct"].isna().all():
    print("[!] No overhead data. Run evaluate.py without --skip_complexity.")
else:
    overhead = float(default_row["overhead_pct"].iloc[0])
    s1_ms    = float(df[(df["run"] == "default")].dropna(subset=["ms_per_img"])["ms_per_img"].iloc[0])                if "ms_per_img" in df.columns else None
    full_ms  = s1_ms  # placeholder; actual comes from complexity dict

    status = "PASS ✓ (<15%)" if overhead < 15 else "FAIL ✗ (≥15%)"
    print(f"  FCM cascade overhead: {overhead:.1f}%  →  {status}")
    print()
    print("  Requirement: <15% inference overhead for the bonus condition")
    print("  (Lecturer: inverted bottleneck FCM should keep overhead minimal)")


## Per-Class AP Analysis

> Run the cell below after training completes.
> It requires `logs/default/eval_results.json` to contain per-class AP
> (torchmetrics returns `map_per_class` when classes are labelled).
> If not present, this cell will gracefully skip.


In [ ]:
# ── Per-Class AP (Stage 1 vs Stage 3) ─────────────────────────────────────
import json

VOC_CLASSES = [
    "aeroplane","bicycle","bird","boat","bottle","bus","car","cat",
    "chair","cow","diningtable","dog","horse","motorbike","person",
    "pottedplant","sheep","sofa","train","tvmonitor",
]

def load_per_class(run="default"):
    path = os.path.join(LOGS_DIR, run, "eval_results.json")
    if not os.path.exists(path):
        return None
    with open(path) as f:
        data = json.load(f)
    return data.get("performance", {})

perf = load_per_class("default")
if perf is None:
    print("[!] logs/default/eval_results.json not found.")
elif "map_per_class" not in perf.get("1", {}):
    print("[!] per-class AP not in results (torchmetrics may not return it for VOC).")
    print("    Use the mAP values from the summary table instead.")
else:
    s1_pc = perf["1"]["map_per_class"]
    s3_pc = perf.get("3", {}).get("map_per_class", [])
    x = np.arange(len(VOC_CLASSES))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - 0.2, [v*100 for v in s1_pc], 0.4, label="Stage 1", alpha=0.85, color="#2196F3")
    ax.bar(x + 0.2, [v*100 for v in s3_pc], 0.4, label="Stage 3", alpha=0.85, color="#FF5722")
    ax.set_xticks(x)
    ax.set_xticklabels(VOC_CLASSES, rotation=45, ha="right")
    ax.set_ylabel("AP (%)")
    ax.set_title("Per-Class AP: Stage 1 vs Stage 3", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(LOGS_DIR, "per_class_ap.png"), bbox_inches="tight")
    plt.show()
